In [1]:
from pathlib import Path

LOCAL_TRAIN_SAMPLES_DIR = Path("../data/train_samples")
LOCAL_VALID_SAMPLES_DIR = Path("../data/valid_samples")

OUTPUT_DIR = Path("../data/output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [2]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ Memory Growth habilitado para: {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"Erro ao configurar memória: {e}")

2025-11-18 17:34:07.492210: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-18 17:34:07.498599: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763498047.506488   88812 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763498047.509416   88812 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763498047.516073   88812 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

✅ Memory Growth habilitado para: 1 GPU(s)


In [3]:
### Data Parameters
BANDS = ['green_median', 'red_median', 'nir_median', 'swir1_median', 'swir2_median', 'ndvi_median', 'ndwi_median']
RESPONSE = ["label"]
FEATURES = BANDS + RESPONSE

### Model Hyperparameters
EPOCHS = 50
BATCH_SIZE = 16

KERNEL_SIZE  = 256
KERNEL_SHAPE = [KERNEL_SIZE, KERNEL_SIZE]
COLUMNS = [tf.io.FixedLenFeature(shape=KERNEL_SHAPE, dtype=tf.float32) for k in FEATURES]

FEATURES_DICT = dict(zip(FEATURES, COLUMNS))

In [4]:
def parse_tfrecord(example_proto):
  """
  The parsing function.
  Read a serialized example into the structure defined by FEATURES_DICT.
  Args:
    example_proto: a serialized Example.
  Returns:
    A dictionary of tensors, keyed by feature name.
  """
  return tf.io.parse_single_example(example_proto, FEATURES_DICT)


def to_tuple(inputs):
  """
  Function to convert a dictionary of tensors to a tuple of (inputs, outputs).
  Turn the tensors returned by parse_tfrecord into a stack in HWC shape.
  Args:
    inputs: A dictionary of tensors, keyed by feature name.
  Returns:
    A dtuple of (inputs, outputs).
  """
  inputsList = [inputs.get(key) for key in FEATURES]
  
  stacked = tf.stack(inputsList, axis=0)
  stacked = tf.transpose(stacked, [1, 2, 0]) # Convert from CHW to HWC
    
  return stacked[:,:,:len(BANDS)], stacked[:,:,len(BANDS):]


def get_dataset(pattern):
  """
  Function to read, parse and format to tuple a set of input tfrecord files.
  Get all the files matching the pattern, parse and convert to tuple.
  Args:
    pattern: A file pattern to match in a Cloud Storage bucket.
  Returns:
    A tf.data.Dataset
  """
  glob = tf.io.gfile.glob(pattern)
  
  dataset = tf.data.TFRecordDataset(glob, compression_type='GZIP', num_parallel_reads=16)
  dataset = dataset.map(parse_tfrecord, num_parallel_calls=16)
  dataset = dataset.map(to_tuple, num_parallel_calls=16)
    
  return dataset

In [5]:
def squeeze_mask(image, mask):
  """Ensures the mask has the shape [H, W] and not [H, W, 1]."""
  with tf.device('/gpu:0'):
      mask = tf.squeeze(mask, axis=-1)
      mask = tf.cast(mask, tf.uint8)

      return image, mask

def augment_spatial(image, label):
    """Randomly translates/pads the image."""

    with tf.device('/gpu:0'):
        label = label[..., tf.newaxis] # Temporarily add back the channel: (256, 256, 1)

        padded_image = tf.pad(image, [[64, 64], [64, 64], [0, 0]], mode='CONSTANT')
        padded_label = tf.pad(label, [[64, 64], [64, 64], [0, 0]], mode='CONSTANT')

        random_x = tf.random.uniform([], 0, 129, dtype=tf.int32)
        random_y = tf.random.uniform([], 0, 129, dtype=tf.int32)

        cropped_image = tf.slice(padded_image, [random_x, random_y, 0], [256, 256, len(BANDS)])
        cropped_label = tf.slice(padded_label, [random_x, random_y, 0], [256, 256, 1])

        cropped_label = tf.squeeze(cropped_label, axis=-1)

        return cropped_image, cropped_label

def removeNan(image, label):
    image = tf.keras.ops.nan_to_num(image)
    label = tf.keras.ops.nan_to_num(label)
    
    return image, label

def get_training_dataset():
    pattern = str(LOCAL_TRAIN_SAMPLES_DIR / '*.tfrecord.gz')
    
    dataset = get_dataset(pattern)

    squeezed_dataset = dataset.map(squeeze_mask, num_parallel_calls=tf.data.AUTOTUNE)
    augmented_dataset = squeezed_dataset.map(augment_spatial, num_parallel_calls=tf.data.AUTOTUNE)
    
    final_dataset = augmented_dataset.concatenate(squeezed_dataset)
    final_dataset = final_dataset.shuffle(20000, reshuffle_each_iteration=True).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
    
    return final_dataset.map(removeNan, num_parallel_calls=tf.data.AUTOTUNE)

In [6]:
def squeeze_and_cast_mask(image, mask):
    """Ensures the mask has the shape [H, W] and not [H, W, 1]."""
    with tf.device('/gpu:0'):
        mask = tf.squeeze(mask, axis=-1)
        mask = tf.cast(mask, tf.uint8)
        return image, mask


def get_eval_dataset():
    """
    Creates the evaluation dataset pipeline.
    NO augmentation, NO concatenation, NO shuffle, NO repeat.
    """
    pattern = str(LOCAL_VALID_SAMPLES_DIR / '*.tfrecord.gz')

    # 1. Get the initial dataset
    dataset = get_dataset(pattern)

    # 2. Apply the squeeze and cast operation to ensure correct mask shape
    dataset = dataset.map(squeeze_and_cast_mask, num_parallel_calls=tf.data.AUTOTUNE)

    # 3. Batch the data and prefetch for performance
    # The BATCH_SIZE here can be the same or different from your training batch size
    final_dataset = dataset.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

    return final_dataset.map(removeNan, num_parallel_calls=tf.data.AUTOTUNE)

# Validação das amostras

In [ ]:
def squeeze_mask(image, mask):
    """Ensures the mask has the shape [H, W] and not [H, W, 1]."""
    mask = tf.squeeze(mask, axis=-1)

    return image, mask


def run_integrity_check(name, file_glob):
    """
    Iterates through an entire dataset to find shape inconsistencies AND invalid numerical values (NaN or Inf).
    """
    print(f"\n--- Starting Full Integrity Check for: {name} ---")

    # 1. Create a dataset pipeline WITHOUT shuffle, batch, or prefetch
    raw_dataset = get_dataset(file_glob)
    # NOTE: We cast to float32 here to check for NaN/Inf before any other processing
    processed_dataset = raw_dataset.map(lambda img, msk: (tf.cast(img, tf.float32), tf.cast(msk, tf.float32)))

    error_found = False
    # 2. Enumerate allows us to see which example number is bad
    for i, (image, mask) in enumerate(processed_dataset):

        # --- NEW: Check for NaN or Inf values ---
        if tf.reduce_any(tf.math.is_nan(image)):
            print(f"!!! NaN VALUE FOUND in IMAGE at example index: {i} !!!")
            error_found = True
            break
        if tf.reduce_any(tf.math.is_inf(image)):
            print(f"!!! Inf VALUE FOUND in IMAGE at example index: {i} !!!")
            error_found = True
            break
        if tf.reduce_any(tf.math.is_nan(mask)):
            print(f"!!! NaN VALUE FOUND in MASK at example index: {i} !!!")
            error_found = True
            break
        if tf.reduce_any(tf.math.is_inf(mask)):
            print(f"!!! Inf VALUE FOUND in MASK at example index: {i} !!!")
            error_found = True
            break
        # --- End of new check ---

        # Squeeze the mask for shape checking
        image, mask = squeeze_mask(image, mask)

        image_shape = image.shape
        mask_shape = mask.shape

        # 3. Check if shapes are correct for every single example
        if len(image_shape) != 3 or image_shape[0] != 256 or image_shape[1] != 256 or image_shape[2] != len(BANDS):
            print(f"!!! CORRUPTED IMAGE SHAPE FOUND at example index: {i} !!!")
            print(f"    Expected shape: (256, 256, 7), but got: {image_shape}")

            error_found = True

            break

        if len(mask_shape) != 2 or mask_shape[0] != 256 or mask_shape[1] != 256:
            print(f"!!! CORRUPTED MASK SHAPE FOUND at example index: {i} !!!")
            print(f"    Expected shape: (256, 256), but got: {mask_shape}")

            error_found = True

            break

    if error_found:
        print(f"--- {name} dataset check FAILED. ---")
    else:
        print(f"--- Full {name} dataset check PASSED. ---")


# --- Run the check on both your datasets ---
run_integrity_check("Training Data", str(LOCAL_TRAIN_SAMPLES_DIR / '*.tfrecord.gz'))
run_integrity_check("Validate Data", str(LOCAL_VALID_SAMPLES_DIR / '*.tfrecord.gz'))


--- Starting Full Integrity Check for: Training Data ---
--- Full Training Data dataset check PASSED. ---

--- Starting Full Integrity Check for: Validate Data ---
--- Full Validate Data dataset check PASSED. ---


2025-11-18 17:36:16.658298: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [9]:
print("\n--- Verifying Label Range ---")

raw_training_data = get_dataset(str(LOCAL_TRAIN_SAMPLES_DIR / '*.tfrecord.gz'))
squeezed_training_data = raw_training_data.map(squeeze_mask)

min_label = 999
max_label = -999

for i, (image, mask) in enumerate(squeezed_training_data):
    current_min = tf.reduce_min(mask)
    current_max = tf.reduce_max(mask)

    if current_min.numpy() < min_label:
        min_label = current_min

    if current_max.numpy() > max_label:
        max_label = current_max

print(f"\nScan Complete. Min: {min_label}, Max: {max_label}")


--- Verifying Label Range ---

Scan Complete. Min: 0.0, Max: 1.0


# Construção do modelo

In [10]:
from pathlib import Path

OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoint'
CHECKPOINT_DIR.mkdir(exist_ok=True)

LOGGER_DIR = OUTPUT_DIR / 'logs'
LOGGER_DIR.mkdir(exist_ok=True)

In [11]:
from keras.models import Model
from keras.layers import Input, Conv2D, UpSampling2D
from keras.applications import MobileNetV3Large

from matplotlib import pyplot as plt

In [12]:
NCLASSES = 2

CONNECTION_POINT_NAME = 'conv_bn'

# --- Part 1: Build the Encoder (your custom feature extractor) ---

mobile_net_v3_large = MobileNetV3Large(include_top=False, input_shape=(256, 256, 3), classes=NCLASSES)
connection_layer = mobile_net_v3_large.get_layer(CONNECTION_POINT_NAME)
tail_model = Model(inputs=connection_layer.input, outputs=mobile_net_v3_large.output, name='mobilenet_tail')

input_layer = Input(shape=(256, 256, len(BANDS)), name=f'{len(BANDS)}_band_input')
first_layer = Conv2D(16, (3, 3), strides=(2, 2), padding='same', name='new_first_conv')(input_layer)
encoder_output = tail_model(first_layer)

encoder = Model(inputs=input_layer, outputs=encoder_output, name='encoder')

# --- Part 2: Build the Decoder (NEW PART) ---

x = encoder.output

for itter in range(5):
    x = UpSampling2D(size=(2, 2))(x)
    x = Conv2D(512//(2**itter), (3, 3), activation='relu', padding='same')(x)

# --- Part 3: Add the final segmentation head ---

final_output = Conv2D(NCLASSES, (1, 1), activation='softmax', name='segmentation_head')(x)

# --- Part 4: Create and Compile the final model ---

model = Model(inputs=encoder.input, outputs=final_output)

# --- Part 5: Visualize model summary ---

model.summary()# Contrução/Treinamento modelo

/home/tiago/miniforge3/envs/mapbiomas/lib/python3.12/site-packages/keras/src/applications/mobilenet_v3.py:519: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ 7_band_input (InputLayer)       │ (None, 256, 256, 7)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_first_conv (Conv2D)         │ (None, 128, 128, 16)   │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenet_tail (Functional)     │ (None, 8, 8, 960)      │     2,995,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d (UpSampling2D)    │ (None, 16, 16, 960)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 16, 16, 512)    │     4,424,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_1 (UpSampling2D)  │ (None, 32, 32, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 256)    │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_2 (UpSampling2D)  │ (None, 64, 64, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 128)    │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_3 (UpSampling2D)  │ (None, 128, 128, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 128, 128, 64)   │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_4 (UpSampling2D)  │ (None, 256, 256, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 256, 256, 32)   │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ segmentation_head (Conv2D)      │ (None, 256, 256, 2)    │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,988,402 (34.29 MB)

 Trainable params: 8,964,002 (34.19 MB)

 Non-trainable params: 24,400 (95.31 KB)

In [16]:
# --- THE ISOLATION EXPERIMENT ---
modelCheckpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=CHECKPOINT_DIR / "cp-{epoch:04d}.keras",
    monitor='val_accuracy',
    save_weights_only=False,
    save_best_only=True,
    save_freq=5,
    verbose=1,
    )

earlyStopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    min_delta=0.01,
    patience=10,
    restore_best_weights=True,
    verbose=1,
    )

metricIoU = tf.keras.metrics.IoU(num_classes=2, target_class_ids=[1])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
    jit_compile=False
    )

In [17]:
train_dataset = get_training_dataset()
valid_dataset = get_eval_dataset()

In [18]:
with tf.device('/gpu:0'):
    result = model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=valid_dataset,
        callbacks=[earlyStopping, modelCheckpoint]
    )

Epoch 1/50
      4/Unknown 29s 229ms/step - accuracy: 0.4707 - loss: 1.2074

/home/tiago/miniforge3/envs/mapbiomas/lib/python3.12/site-packages/keras/src/callbacks/model_checkpoint.py:276: UserWarning: Can save best model only with val_accuracy available.
  if self._should_save_model(epoch, batch, logs, filepath):


     80/Unknown 59s 393ms/step - accuracy: 0.7355 - loss: 0.5686

2025-11-18 17:50:40.672014: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2025-11-18 17:50:40.673543: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 135995859995216228
2025-11-18 17:50:40.673554: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 13868401961689054292
2025-11-18 17:50:40.673556: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 6568886833003758952
/home/tiago/miniforge3/envs/mapbiomas/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warn

80/80 ━━━━━━━━━━━━━━━━━━━━ 62s 432ms/step - accuracy: 0.7872 - loss: 0.4545 - val_accuracy: 0.8488 - val_loss: 1.7048
Epoch 2/50


...

2025-11-18 17:55:57.369637: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 1024753628569915832
2025-11-18 17:55:57.369662: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 16698932860048018782
2025-11-18 17:55:57.369665: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 13483340682808028443
2025-11-18 17:55:59.219744: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 33554432 bytes after encountering the first element of size 33554432 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 385ms/step - accuracy: 0.8696 - loss: 0.2838

2025-11-18 17:56:30.126672: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 135995859995216228
2025-11-18 17:56:30.126703: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 13868401961689054292
2025-11-18 17:56:30.126705: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 6568886833003758952


80/80 ━━━━━━━━━━━━━━━━━━━━ 34s 402ms/step - accuracy: 0.8701 - loss: 0.2844 - val_accuracy: 0.8489 - val_loss: 0.3632
Epoch 11: early stopping
Restoring model weights from the end of the best epoch: 1.


2025-11-18 17:56:31.450111: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 1024753628569915832
2025-11-18 17:56:31.450137: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 16698932860048018782
2025-11-18 17:56:31.450140: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 13483340682808028443


In [ ]:
def previewClass(model, n_samples):
    validation = get_eval_dataset()

    for batch in validation.shuffle(100).take(n_samples):
        pureImage = batch[0]
        supervised = batch[1]

        stacked = tf.transpose(pureImage[0], [0, 1, 2]).numpy()
        stackedS = tf.transpose(supervised[0], [0, 1]).numpy()

        test_pred_raw = model.predict(pureImage)
        test_pred_raw = tf.argmax(test_pred_raw, axis=-1)
        test_pred_raw = tf.transpose(test_pred_raw[0],[0, 1]).numpy()

        fig = plt.figure(figsize=[12,4])

        # Visualização da imagem por bandas
        fig.add_subplot(131)
        plt.imshow(stacked[:,:,0:3].astype(np.uint8), interpolation='nearest', vmin=0, vmax=255)

        # Visualização dos labels em cinza
        fig.add_subplot(132)
        plt.imshow(stackedS[:,:], interpolation='nearest',cmap="gray")

        # Visualização da segmentação feita pelo modelo
        fig.add_subplot(133)
        plt.imshow(test_pred_raw[:,:], interpolation='nearest',cmap="gray")

        plt.show()

previewClass(model, 4)

# Salvar modelo

In [ ]:
TARGET_LAYER_NAME = 'conv2d_2'

try:
    input_tensor = model.input
    output_tensor = model.get_layer(TARGET_LAYER_NAME).output

    new_feature_extractor = Model(
        inputs=input_tensor,
        outputs=output_tensor,
        name=f"extractor_until_{TARGET_LAYER_NAME}"
    )

    print("--- Successfully created new extractor ---")
    new_feature_extractor.summary()

    new_feature_extractor.save('my_model.keras')

except ValueError as e:
    print(f"\n--- ERROR ---")
    print(f"Could not find a layer named '{TARGET_LAYER_NAME}'.")
    print("Please check the name in your model summary and try again.")